In [1]:
import time
import pandas as pd
from transformers import pipeline

c:\Users\basma\ai_engineer_starter_kit\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 15-sentence evaluation set: mix of clearly toxic, clearly non-toxic, and a few borderline/sarcastic cases.
# label: 1 = toxic (ground truth, labeled by us for accuracy scoring), 0 = non-toxic
test_set = [
    ("You are an absolute idiot and you should be ashamed of yourself.", 1),
    ("I really appreciate your help with this, thank you so much!", 0),
    ("Shut up, nobody asked for your opinion, get lost.", 1),
    ("This documentation is well written and easy to follow.", 0),
    ("You people are all the same, worthless and pathetic.", 1),
    ("Let's meet at 10am tomorrow to discuss the project timeline.", 0),
    ("I hope something bad happens to you, you deserve it.", 1),
    ("Great job on the presentation, the team loved it.", 0),
    ("Stop being such a coward and just admit you were wrong.", 1),
    ("The weather today is sunny with a light breeze.", 0),
    ("Your code is garbage and so are you honestly.", 1),
    ("Congratulations on finishing the certification, well deserved!", 0),
    ("Nobody likes you, just disappear already.", 1),
    ("Can you share the updated spreadsheet when you get a chance?", 0),
    ("Wow, what a genius move, said no one ever, you clown.", 1),
]

sentences = [s for s, _ in test_set]
ground_truth = [label for _, label in test_set]
len(sentences)

15

In [3]:
candidate_models = {
    "unitary/toxic-bert": "unitary/toxic-bert",
    "martin-ha/toxic-comment-model": "martin-ha/toxic-comment-model",
    "s-nlp/roberta_toxicity_classifier": "s-nlp/roberta_toxicity_classifier",
}

In [4]:
def normalize_prediction(model_name: str, result: dict) -> int:
    """Map each model's raw output label to a common 0/1 toxic label.

    unitary/toxic-bert returns labels like 'toxic' with a score; treat score > 0.5 on the
    'toxic' label as toxic.
    martin-ha/toxic-comment-model returns 'toxic' / 'non-toxic'.
    s-nlp/roberta_toxicity_classifier returns 'toxic' / 'neutral'.
    """
    label = result["label"].lower()
    if "non" in label or "neutral" in label or label in ("label_0", "clean"):
        return 0
    return 1

In [ ]:
rows = []

for display_name, hf_id in candidate_models.items():
    print(f"Loading {display_name} ...")
    clf = pipeline("text-classification", model=hf_id, top_k=None)

    correct = 0
    total_time = 0.0

    for sentence, true_label in zip(sentences, ground_truth):
        start = time.perf_counter()
        output = clf(sentence)
        elapsed = time.perf_counter() - start
        total_time += elapsed

        # output is a list of {label, score} dicts (all labels) or a single dict depending on the model
        top_result = max(output[0], key=lambda x: x["score"]) if isinstance(output[0], list) else output[0]
        predicted_label = normalize_prediction(display_name, top_result)

        if predicted_label == true_label:
            correct += 1

    accuracy = correct / len(sentences)
    avg_latency_ms = (total_time / len(sentences)) * 1000

    rows.append({
        "model": display_name,
        "accuracy": accuracy,
        "avg_latency_ms": round(avg_latency_ms, 1),
    })

results_df = pd.DataFrame(rows)
results_df

Loading unitary/toxic-bert ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4365.67it/s]


Loading martin-ha/toxic-comment-model ...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 12360.66it/s]


Loading s-nlp/roberta_toxicity_classifier ...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 26912.31it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: s-nlp/roberta_toxicity_classifier
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,model,accuracy,avg_latency_ms
0,unitary/toxic-bert,0.533333,53.3
1,martin-ha/toxic-comment-model,0.733333,29.0
2,s-nlp/roberta_toxicity_classifier,1.000000,40.2
